# 🛒 실습: 고객 문의(VOC) 자동 분류 + CS 티켓 라우팅

```
Phase 0~4. Pydantic + RunnableParallel 핵심 부품
Project.   고객 문의 → 개인정보 마스킹 → 분석 + 답변 초안 + 태깅 (동시) → 티켓 라우팅 → 일일 리포트
```

**시나리오**  
당신은 신선식품 이커머스 **FreshMart**(가상)의 CS 자동화 담당자입니다.  
하루 수백 건씩 들어오는 고객 문의를 LLM으로 **분류**하고, **답변 초안**을 만들고, **담당팀·우선순위·SLA**를 자동 배정하는 파이프라인을 만드세요.

**작성 방법**
- `# TODO` 와 `____` 부분을 채우세요.
- 각 문제 아래의 **✅ 체크 셀**을 실행해 통과하면 다음 단계로 넘어가세요.
- 막히면 💡 힌트를 참고하세요. 

## Phase 0. 환경 설정 (실행만 하세요)

In [ ]:
%pip install -q langchain langchain-openai python-dotenv pydantic

In [ ]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

# 2️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


import re
from collections import Counter
from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Phase 1. PydanticOutputParser — 고객 문의를 객체로

### 문제 1. 문의 분석 체인 만들기
5단계 패턴: **스키마 → 파서 → 프롬프트 → LLM → 체인**

`InquiryAnalysis` 스키마 요구사항

| 필드 | 타입 | 설명 |
|---|---|---|
| `category` | `"배송"`, `"환불/교환"`, `"상품품질"`, `"결제"`, `"회원/계정"`, `"기타"` 중 하나 | 문의 유형 |
| `sentiment` | `"positive"`, `"negative"`, `"neutral"` 중 하나 | 고객 감정 |
| `urgency` | 정수 1~5 | 긴급도 (범위 검증 필수) |
| `order_id` | 문자열 또는 `None` | 주문번호(`FM-`로 시작), 없으면 `None` |
| `summary` | 문자열 | 한 줄 요약 |

💡 힌트: `Literal[...]`, `Field(ge=, le=)`, `Optional[str]`, `.partial(format_instructions=parser.get_format_instructions())`

In [ ]:
INQUIRY = """주문번호 FM-20261005-1182 입니다. 어제 받은 딸기 한 박스가 절반 이상 물러서 먹을 수가 없네요.
사진 첨부했습니다. 환불이나 재배송 해주세요. 이번이 두 번째라 정말 실망입니다."""

# TODO 1-1: 스키마 정의
class InquiryAnalysis(BaseModel):
    category: ____ = Field(description="문의 유형")
    sentiment: ____
    urgency: int = Field(____, description="긴급도 1(낮음)~5(매우 높음)")
    order_id: ____ = Field(default=None, description="주문번호(FM-로 시작). 없으면 null")
    summary: str = Field(description="20자 이내 한 줄 요약")

# TODO 1-2: 파서 생성
parser_inquiry = ____

# TODO 1-3: 프롬프트 (format_instructions 주입)
prompt_inquiry = ChatPromptTemplate.from_messages([
    ("system", "당신은 FreshMart(신선식품 이커머스) CS 분석가입니다. 고객 문의를 분석하세요. 한국어로.\n\n{format_instructions}"),
    ("human", "{inquiry}"),
]).____(format_instructions=____)

# TODO 1-4: 체인 조립 (프롬프트 → LLM → 파서)
chain_analysis = ____

r = chain_analysis.invoke({"inquiry": INQUIRY})
print(r)

In [ ]:
# ✅ 체크 1
assert isinstance(r, InquiryAnalysis), "결과가 InquiryAnalysis 객체가 아닙니다"
assert 1 <= r.urgency <= 5, "urgency 범위 오류"
assert r.order_id == "FM-20261005-1182", f"주문번호 추출 확인: {r.order_id}"
assert InquiryAnalysis.model_fields["urgency"].metadata, "urgency에 ge/le 검증이 없습니다"
print("✅ 문제 1 통과")

## Phase 2. RunnableParallel — 분석 + 답변 초안 동시 실행

### 문제 2. 답변 초안 체인 + 결합
- `chain_reply`: 고객에게 보낼 **답변 초안**(자연어, `StrOutputParser`)
  - 조건: 존댓말, 3~4문장, **사과 → 조치 안내 → 마무리 인사** 순서, 확정되지 않은 보상 약속 금지
- `cs_v1`: `analysis`와 `reply` 두 체인을 **동시에** 실행

💡 힌트: `RunnableParallel(키1=체인1, 키2=체인2)`

In [ ]:
# TODO 2-1: 답변 초안 체인
chain_reply = (
    ChatPromptTemplate.from_messages([
        ("system", ____),   # 조건을 프롬프트로 작성
        ("human", "{inquiry}"),
    ]) | llm | ____
)

# TODO 2-2: 분석 + 답변 동시 실행
cs_v1 = RunnableParallel(____)

r = cs_v1.invoke({"inquiry": INQUIRY})
print(r["analysis"])
print("-" * 40)
print(r["reply"])

In [ ]:
# ✅ 체크 2
assert set(r.keys()) == {"analysis", "reply"}, f"키 확인: {r.keys()}"
assert isinstance(r["analysis"], InquiryAnalysis)
assert isinstance(r["reply"], str) and len(r["reply"]) > 20
print("✅ 문제 2 통과")

## Phase 3. RunnableLambda — 규칙 기반 티켓 라우팅

### 문제 3. 분석 결과 → CS 티켓 (결과는 Pydantic)

**담당팀**: `TEAM_MAP`으로 category → 팀 매핑 (없으면 `"CS팀"`)

**우선순위 규칙** (위에서부터 먼저 걸리는 규칙 적용)

| 조건 | priority | sla_hours | title |
|---|---|---|---|
| urgency ≥ 4 **그리고** sentiment == negative | `P1` | 2 | `"[긴급] " + summary` |
| urgency ≥ 2 | `P2` | 24 | summary |
| 그 외 | `P3` | 72 | summary |

> LLM은 판단(분류)만, **업무 규칙은 코드로** — 규칙이 바뀌어도 프롬프트를 건드리지 않습니다.

In [ ]:
TEAM_MAP = {
    "배송": "물류팀",
    "환불/교환": "CS팀",
    "상품품질": "품질관리팀",
    "결제": "결제팀",
    "회원/계정": "CS팀",
    "기타": "CS팀",
}

# TODO 3-1: 티켓 스키마
class Ticket(BaseModel):
    priority: ____
    team: str
    sla_hours: int = Field(description="응답 기한(시간)")
    title: str

# TODO 3-2: 라우팅 함수
def make_ticket(x) -> Ticket:
    a = x["analysis"]
    team = ____
    if ____:
        return Ticket(priority="P1", team=team, sla_hours=2, title=f"[긴급] {a.summary}")
    if ____:
        return ____
    return ____

# TODO 3-3: cs_v1 뒤에 라우팅 연결
cs_v2 = cs_v1 | ____

print(cs_v2.invoke({"inquiry": INQUIRY}))

In [ ]:
# ✅ 체크 3 — LLM 없이 규칙만 단위 테스트
def _fake(category, sentiment, urgency):
    return {"analysis": InquiryAnalysis(category=category, sentiment=sentiment,
                                        urgency=urgency, order_id=None, summary="테스트"),
            "reply": ""}

t = make_ticket(_fake("상품품질", "negative", 5))
assert (t.priority, t.team, t.sla_hours) == ("P1", "품질관리팀", 2), t
assert t.title.startswith("[긴급]")
assert make_ticket(_fake("배송", "neutral", 4)).priority == "P2"     # negative가 아니면 P1 아님
assert make_ticket(_fake("결제", "negative", 3)).team == "결제팀"
t = make_ticket(_fake("기타", "positive", 1))
assert (t.priority, t.sla_hours) == ("P3", 72), t
print("✅ 문제 3 통과")

## Phase 4. RunnablePassthrough — 원문 보존

### 문제 4. 티켓에 원문 첨부하기
상담원은 요약만이 아니라 **고객 원문**도 봐야 합니다. `analysis`, `reply`와 함께 입력 원본을 `original` 키로 그대로 통과시키세요.

In [ ]:
# TODO 4-1
cs_v3 = RunnableParallel(
    analysis=chain_analysis,
    reply=chain_reply,
    original=____,
)

r = cs_v3.invoke({"inquiry": INQUIRY})
print(r["original"])

In [ ]:
# ✅ 체크 4
assert r["original"] == {"inquiry": INQUIRY}
print("✅ 문제 4 통과")

---
# 🛒 Project. FreshMart CS 자동화 파이프라인

```
고객 문의 → [P-1] 개인정보 마스킹 (RunnableLambda)
          → [P-2] 분석 + 답변 초안 + 태깅 + 원문 (RunnableParallel, 동시)
          → [P-3] 최종 티켓 생성 (RunnableLambda + Pydantic)
          → [P-4] 하루치 문의 batch + 일일 리포트
          → [P-5] Slack 알림 메시지 포맷
```

### P-1. 개인정보(PII) 마스킹 — LLM에 보내기 전에
실무에서는 외부 LLM API로 **전화번호·이메일을 그대로 보내면 안 됩니다.** 정규식으로 먼저 가리세요.

| 원본 | 마스킹 결과 |
|---|---|
| `010-2345-6789` / `01023456789` | `010-****-6789` |
| `kimfresh@naver.com` | `ki***@naver.com` |
| `FM-20261005-1182` (주문번호) | **그대로 유지** |

💡 힌트: `re.sub(패턴, r"\1-****-\3", text)` — 그룹 참조 `\1`, `\3`

In [ ]:
PHONE_RE = re.compile(r"(?<!\d)(01[016789])-?(\d{3,4})-?(\d{4})(?!\d)")
EMAIL_RE = re.compile(r"([A-Za-z0-9._%+-]{2})[A-Za-z0-9._%+-]*@")

# TODO P-1: 마스킹 함수 (입력 dict → 출력 dict, 키 이름 유지)
def mask_pii(x: dict) -> dict:
    text = x["inquiry"]
    text = PHONE_RE.sub(____, text)
    text = EMAIL_RE.sub(____, text)
    return ____

print(mask_pii({"inquiry": "연락처 01023456789, 메일 kimfresh@naver.com"}))

In [ ]:
# ✅ 체크 P-1
t = mask_pii({"inquiry": "연락처 010-2345-6789, 01098765432, 메일 kimfresh@naver.com, 주문 FM-20261005-1182"})["inquiry"]
assert "010-****-6789" in t, t
assert "010-****-5432" in t, t
assert "ki***@naver.com" in t, t
assert "FM-20261005-1182" in t, "주문번호는 마스킹되면 안 됩니다"
print("✅ P-1 통과:", t)

### P-2. 이슈 태깅 체인 — 다중 선택 `list[Literal]`
`VOCTags` 스키마
- `issues`: `"신선도"`, `"포장"`, `"배송지연"`, `"오배송"`, `"결제오류"`, `"앱오류"`, `"칭찬"`, `"기타"` 중 **복수 선택**
- `needs_compensation`: 보상(환불·재배송·쿠폰) 검토가 필요한지 `bool`

In [ ]:
# TODO P-2-1: 스키마
class VOCTags(BaseModel):
    issues: ____ = Field(description="문의에 포함된 이슈 (복수 선택)")
    needs_compensation: ____ = Field(description="보상(환불·재배송·쿠폰) 검토 필요 여부")

# TODO P-2-2: 파서 + 체인
parser_tags = ____
chain_tags = (
    ChatPromptTemplate.from_messages([
        ("system", "FreshMart 고객 문의에 이슈 태그를 붙이고 보상 검토 필요 여부를 판단하세요.\n\n{format_instructions}"),
        ("human", "{inquiry}"),
    ]).partial(format_instructions=____)
    | llm | ____
)

print(chain_tags.invoke({"inquiry": INQUIRY}))

In [ ]:
# ✅ 체크 P-2
tags = chain_tags.invoke({"inquiry": INQUIRY})
assert isinstance(tags, VOCTags)
assert "신선도" in tags.issues, tags
assert tags.needs_compensation is True, tags
print("✅ P-2 통과")

### P-3. 전체 파이프라인 조립 + 최종 티켓

**`FinalTicket` 스키마**: `ticket: Ticket`, `issues: list[str]`, `needs_compensation: bool`, `reply_draft: str`, `masked_text: str`

**추가 규칙**: 보상 검토가 필요한데(`needs_compensation=True`) 우선순위가 `P3`이면 → **`P2`, SLA 24시간으로 상향**

**조립 순서**: `mask_pii` → `RunnableParallel(analysis, reply, tags, original)` → `build_final`

💡 `masked_text`는 `original`에서 꺼내면 됩니다. (마스킹 **후** 원문이 통과되므로)  
💡 Pydantic 객체 일부만 바꾸기: `ticket.model_copy(update={...})`

In [ ]:
# TODO P-3-1: 최종 티켓 스키마
class FinalTicket(BaseModel):
    ____

# TODO P-3-2: 최종 티켓 생성 함수 (make_ticket 재사용)
def build_final(x) -> FinalTicket:
    ticket = make_ticket(x)
    if ____:
        ticket = ____
    return FinalTicket(
        ticket=ticket,
        issues=____,
        needs_compensation=____,
        reply_draft=____,
        masked_text=____,
    )

# TODO P-3-3: 파이프라인 조립
cs_pipeline = (
    RunnableLambda(____)
    | RunnableParallel(
        ____
    )
    | RunnableLambda(____)
)

final = cs_pipeline.invoke({"inquiry": INQUIRY})
print(final.model_dump_json(indent=2))

In [ ]:
# ✅ 체크 P-3
assert isinstance(final, FinalTicket)
assert final.ticket.priority in ("P1", "P2")

# 규칙 단위 테스트: 보상 필요 + P3 → P2 상향
x = _fake("기타", "neutral", 1)
x["tags"] = VOCTags(issues=["앱오류"], needs_compensation=True)
x["original"] = {"inquiry": "테스트"}
f = build_final(x)
assert (f.ticket.priority, f.ticket.sla_hours) == ("P2", 24), f.ticket
print("✅ P-3 통과")

### P-4. 하루치 문의 batch + 일일 리포트
- `cs_pipeline.batch(...)`로 5건을 처리하세요. 동시 호출 수는 **3**으로 제한 (`config={"max_concurrency": 3}`) — API rate limit 대응
- 결과로 **우선순위별 건수**, **팀별 건수**, **P1 목록**을 출력하세요.

💡 `Counter(f.ticket.priority for f in results)`

In [ ]:
INQUIRIES = [
    {"inquiry": INQUIRY},
    {"inquiry": "배송 예정일이 10/5였는데 아직도 안 왔어요. 아이 이유식 재료라 급합니다. 010-2345-6789로 바로 연락 주세요."},
    {"inquiry": "결제는 됐는데 주문 내역에 안 보여요. 카드값이 두 번 빠진 것 같습니다. kimfresh@naver.com 으로 내역 보내주세요."},
    {"inquiry": "지난번 배송 기사님이 비 오는 날에도 너무 친절하셨어요. 감사 인사 꼭 전해주세요!"},
    {"inquiry": "앱에서 쿠폰 적용 버튼이 안 눌려요. 급한 건 아니고 다음 주문 때 쓰려고요."},
]

# TODO P-4-1: batch 실행 (max_concurrency=3)
results = ____

# TODO P-4-2: 일일 리포트
ICON = {"P1": "🚨", "P2": "📋", "P3": "📁"}
for f in sorted(results, key=lambda f: f.ticket.priority):
    print(f"{ICON[f.ticket.priority]} [{f.ticket.priority}] {f.ticket.team:<6} | {f.ticket.title}")

print("\n📊 우선순위별:", ____)
print("📊 팀별:", ____)
print("🚨 P1 목록:", ____)

In [ ]:
# ✅ 체크 P-4
assert len(results) == 5 and all(isinstance(f, FinalTicket) for f in results)
assert all("010-2345-6789" not in f.masked_text for f in results), "전화번호가 마스킹되지 않았습니다"
assert all("kimfresh@" not in f.masked_text for f in results), "이메일이 마스킹되지 않았습니다"
assert results[3].ticket.priority != "P1", "칭찬 문의가 P1으로 분류되면 안 됩니다"
assert any(f.ticket.priority == "P1" for f in results), "P1이 최소 1건은 있어야 합니다"
print("✅ P-4 통과")

### P-5. Slack 알림 메시지 포맷
`FinalTicket` → Slack에 올릴 문자열로 변환하세요. 보상 검토 대상이면 `💰 보상검토` 표시.

```
🚨 [P1] [긴급] 딸기 품질 불량, 재발 고객
담당: 품질관리팀 | SLA 2h | 💰 보상검토
이슈: 신선도, 포장
```

In [ ]:
# TODO P-5
def to_slack(f: FinalTicket) -> str:
    ____

for f in results:
    print(to_slack(f))
    print()